# Chapter 04 — APIs, Web Data, and Schema Validation

External systems expose data through HTTP APIs or web pages. Learn HTTP request structure, JSON response handling, defensive parsing, HTML extraction, and schema validation.

## Prerequisites
Dictionaries, JSON, exceptions, and basic DataFrames; live internet access is optional.

## Learning goals
- Explain URLs, methods, status codes, headers, query parameters, and timeouts.
- Use Requests to retrieve and check API responses.
- Parse HTML locally with BeautifulSoup.
- Validate records with Pydantic and report malformed data.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. HTTP and REST
2. Requests API and arguments
3. Error handling and retries
4. BeautifulSoup
5. Pydantic contracts

### HTTP and REST
An HTTP request combines a method (`GET`, `POST`, `PUT`, `PATCH`, `DELETE`), URL, headers, optional query parameters, and sometimes a body. Responses carry status codes, headers, and content. `2xx` means success; `3xx` redirection; `4xx` client-side issue; `5xx` server-side issue. REST-style APIs model resources, but actual APIs differ: check authentication, pagination, rate limits, and terms.

#### Demonstration 1 — Inspect local JSON API fixture

Run the code and inspect both the output and the assumptions behind it.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import json
payload=json.loads((DATA/"sample_api_response.json").read_text(encoding="utf-8"))
print(payload["status"],payload["count"],payload["results"][0])

Data directory: /mnt/data/python_data_analysis_book/data
ok 3 {'id': 1, 'name': 'Notebook', 'price': 120.0}


### Requests API and arguments
`requests.get(url, params=None, headers=None, timeout=None, auth=None, verify=True, allow_redirects=True)` sends a GET. `params` encodes query parameters; `headers` sends metadata; `timeout` bounds waits; `verify=True` validates TLS certificates and should normally stay enabled. Inspect `status_code`, `headers`, `text`, `content`; use `.json()` for JSON and `.raise_for_status()` for unsuccessful HTTP responses.

#### Demonstration 2 — Convert JSON records to a DataFrame

Run the code and inspect both the output and the assumptions behind it.

In [3]:
import pandas as pd
records=payload["results"]; required={"id","name","price"}
assert all(required.issubset(r) for r in records)
products=pd.DataFrame.from_records(records); display(products)

   id      name  price
0   1  Notebook  120.0
1   2       Pen   15.0
2   3  Backpack  850.0


### Error handling and retries
Set timeouts. Catch `Timeout`, `ConnectionError`, and `HTTPError` separately when recovery differs. Respect `Retry-After` for rate limits; bound retries and use backoff for transient failures. Do not retry non-idempotent writes blindly. Validate response shape before loading it into an analytical table. Never log tokens or passwords.

#### Demonstration 3 — Parse offline HTML

Run the code and inspect both the output and the assumptions behind it.

In [4]:
from bs4 import BeautifulSoup
html=(DATA/"sample_page.html").read_text(encoding="utf-8")
soup=BeautifulSoup(html,"html.parser")
items=[{"id":n.get("data-id"),"label":n.get_text(" ",strip=True)} for n in soup.select("li.product")]
display(pd.DataFrame(items))

  id           label
0  1  Notebook - 120
1  2        Pen - 15
2  3  Backpack - 850


### BeautifulSoup
`BeautifulSoup(markup, "html.parser")` parses HTML. `find` returns the first match; `find_all` returns all matches; CSS selectors use `.select` and `.select_one`. `.get_text(separator=" ", strip=True)` extracts clean text; `.get("href")` reads an attribute. Prefer documented APIs; respect site policies and minimize requests.

#### Demonstration 4 — Validate product records

Run the code and inspect both the output and the assumptions behind it.

In [5]:
from pydantic import BaseModel, Field, ValidationError
class Product(BaseModel):
 id:int
 name:str=Field(min_length=1)
 price:float=Field(gt=0)
valid=[]
for r in records:
 try: valid.append(Product.parse_obj(r).dict())
 except ValidationError as e: print(e)
print(valid)

[{'id': 1, 'name': 'Notebook', 'price': 120.0}, {'id': 2, 'name': 'Pen', 'price': 15.0}, {'id': 3, 'name': 'Backpack', 'price': 850.0}]


/mnt/data/python_data_analysis_book/Chapter_04_apis_web_data_and_schema_validation/Chapter_04_Concepts_and_Theory.ipynb:8: PydanticDeprecatedSince20:

The `parse_obj` method is deprecated; use `model_validate` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/

/mnt/data/python_data_analysis_book/Chapter_04_apis_web_data_and_schema_validation/Chapter_04_Concepts_and_Theory.ipynb:8: PydanticDeprecatedSince20:

The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/



### Pydantic contracts
Pydantic models validate records using type annotations. `BaseModel` declares fields; `Field(gt=0, min_length=1, ...)` adds constraints. Validation errors identify invalid fields and reasons. Choose whether invalid data stops a job, is quarantined, or is corrected under an explicit rule. Pydantic v2 uses `model_validate`; `parse_obj` is a common v1/v2 compatibility path but deprecated in v2.

#### Demonstration 5 — Safe live-request pattern (not called)

Run the code and inspect both the output and the assumptions behind it.

In [6]:
import requests
def fetch_json(url, params=None):
    response=requests.get(url,params=params,timeout=(5,20),headers={"Accept":"application/json"})
    response.raise_for_status()
    return response.json()
print("Function defined; call only with a documented API endpoint and required authorization.")

Function defined; call only with a documented API endpoint and required authorization.


## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: HTTP and REST.
- Explain one important concept from: Requests API and arguments.
- Explain one important concept from: Error handling and retries.
- Explain one important concept from: BeautifulSoup.
- Explain one important concept from: Pydantic contracts.